# Moroccan Housing Price Prediction
### End-to-End Machine Learning Project — Mubawab Dataset

Inspired by Chapter 2 of *Hands-On Machine Learning* by Aurélien Géron.  
We follow the same pipeline structure — adapted for the Moroccan real estate market (Mubawab scrape, March 2026).

**Goal:** Predict apartment prices (`price_dh`) across 7 Moroccan cities.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

np.random.seed(42)
print('All libraries loaded.')

## 2. Load the Data

In [ ]:
df = pd.read_csv('dataset_mubawab-housing.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Quick Look at the Data

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Value counts for categorical columns
for col in ['city', 'property_type', 'standing', 'state', 'transaction_type']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False))

In [ ]:
# Missing values summary
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, 'pct': missing_pct})[missing > 0]

In [ ]:
# Histograms of all numeric features
df.hist(bins=50, figsize=(16, 10))
plt.suptitle('Histogram of Numeric Features (Raw Data)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Cleaning — Before Splitting

Steps:
- Remove `location` rows (only 8 — we focus on sales: `vente`)
- Drop duplicate listings (same `listing_id` scraped twice)
- Drop rows with no `price_dh` (target)
- Drop columns that are useless or leaky:
  - `listing_id`, `listing_title`, `listing_url`, `scraped_at` — identifiers/metadata
  - `property_type` — all `appartement`, zero variance
  - `price_per_m2` — derived from price/surface, **leaks the target**

In [ ]:
# Keep only sale listings
df = df[df['transaction_type'].isin(['vente', 'unknown'])].copy()
print(f'After filtering transaction_type: {df.shape}')

# Drop duplicate listing_ids (keep first scrape)
df = df.drop_duplicates(subset='listing_id', keep='first').copy()
print(f'After deduplication: {df.shape}')

# Drop rows with no price (our target)
df = df.dropna(subset=['price_dh']).copy()
print(f'After dropping missing price_dh: {df.shape}')

# Drop leaky and irrelevant columns
drop_cols = ['listing_id', 'listing_title', 'listing_url', 'scraped_at',
             'property_type', 'price_per_m2', 'transaction_type']
df = df.drop(columns=drop_cols)
print(f'After dropping irrelevant columns: {df.shape}')
df.head()

## 5. Create a Test Set — Stratified Split on City

Like Géron uses income category for stratified sampling, we use `city` — the strongest geographic signal in this dataset.

In [ ]:
# Stratified split preserving city distribution
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(df, df['city']):
    strat_train = df.iloc[train_idx].copy()
    strat_test  = df.iloc[test_idx].copy()

print(f'Train set: {strat_train.shape}')
print(f'Test set:  {strat_test.shape}')

# Check city proportions are preserved
print('\nCity proportions (train vs test):')
pd.DataFrame({
    'overall': df['city'].value_counts(normalize=True).round(3),
    'train':   strat_train['city'].value_counts(normalize=True).round(3),
    'test':    strat_test['city'].value_counts(normalize=True).round(3),
})

## 6. Discover and Visualize the Data

Work only on the training set from here.

In [ ]:
housing = strat_train.copy()

In [ ]:
# Price distribution by city
fig, ax = plt.subplots(figsize=(12, 5))
cities = housing['city'].value_counts().index
data_by_city = [housing[housing['city'] == c]['price_dh'].dropna() / 1_000_000 for c in cities]

ax.boxplot(data_by_city, labels=cities, vert=True)
ax.set_title('Price Distribution by City (MAD millions)', fontsize=13)
ax.set_ylabel('Price (millions MAD)')
ax.set_xlabel('City')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1fM'))
plt.tight_layout()
plt.show()

In [ ]:
# Price vs Surface, coloured by city
fig, ax = plt.subplots(figsize=(12, 6))
city_colors = {'casablanca': '#e74c3c', 'marrakech': '#e67e22', 'rabat': '#3498db',
               'tanger': '#2ecc71', 'agadir': '#9b59b6', 'meknes': '#1abc9c', 'oujda': '#f39c12'}

for city, color in city_colors.items():
    subset = housing[housing['city'] == city]
    ax.scatter(subset['surface_m2'], subset['price_dh'] / 1_000_000,
               alpha=0.5, label=city, color=color, edgecolors='none', s=30)

ax.set_xlabel('Surface (m²)')
ax.set_ylabel('Price (millions MAD)')
ax.set_title('Price vs. Surface by City')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix — numeric features vs price
numeric_cols = housing.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = housing[numeric_cols].corr()
print('Correlation with price_dh:')
corr_matrix['price_dh'].sort_values(ascending=False)

In [ ]:
# Scatter matrix of key numeric features
from pandas.plotting import scatter_matrix
key_attrs = ['price_dh', 'surface_m2', 'num_rooms', 'num_bathrooms', 'floor']
scatter_matrix(housing[key_attrs], figsize=(14, 10), alpha=0.3, diagonal='hist')
plt.suptitle('Scatter Matrix — Key Numeric Features', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 7. Feature Engineering

Inspired by Géron's Chapter 2 — create ratio features that may be more informative than raw counts.

In [ ]:
housing['surface_per_room']    = housing['surface_m2'] / housing['num_rooms']
housing['rooms_per_bathroom']  = housing['num_rooms'] / housing['num_bathrooms']
housing['surface_per_bathroom']= housing['surface_m2'] / housing['num_bathrooms']

# Check new correlations
new_cols = ['surface_per_room', 'rooms_per_bathroom', 'surface_per_bathroom']
corr_new = housing[new_cols + ['price_dh']].corr()['price_dh'].sort_values(ascending=False)
print('Correlation of engineered features with price_dh:')
print(corr_new)

## 8. Prepare Data for Machine Learning

Separate features from target, then define pipelines for numeric and categorical columns.

In [ ]:
# Separate features and target — apply same engineering to test set later via pipeline
# We work on the training set here
housing_features = strat_train.drop(columns=['price_dh'])
housing_labels   = strat_train['price_dh'].copy()

print('Features shape:', housing_features.shape)
print('Labels shape:  ', housing_labels.shape)

### 8.1 Custom Feature Engineering Transformer

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class MoroccanHousingFeatureAdder(BaseEstimator, TransformerMixin):
    """Adds ratio features, mirrors Géron's CombinedAttributesAdder."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['surface_per_room']     = X['surface_m2'] / X['num_rooms'].replace(0, np.nan)
        X['rooms_per_bathroom']   = X['num_rooms']  / X['num_bathrooms'].replace(0, np.nan)
        X['surface_per_bathroom'] = X['surface_m2'] / X['num_bathrooms'].replace(0, np.nan)
        return X

### 8.2 Define Feature Columns

In [ ]:
# After feature engineering, these will be the numeric columns
num_features = ['surface_m2', 'num_rooms', 'num_bathrooms', 'floor',
                'surface_per_room', 'rooms_per_bathroom', 'surface_per_bathroom']

# standing has many nulls — treat as a separate low-cardinality categorical
# state, city, neighborhood — OHE
cat_features_ohe = ['city', 'state', 'standing']

# neighborhood has too many unique values — use target encoding or drop for now
# For this pipeline we drop it (can revisit with target encoding later)
cols_to_drop_in_pipeline = ['neighborhood']

print('Numeric features:', num_features)
print('Categorical (OHE):', cat_features_ohe)
print('\nNeighborhood unique values:', housing_features['neighborhood'].nunique())

## 9. Build the Full Sklearn Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Numeric pipeline: impute median → scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Categorical pipeline: impute 'unknown' → one-hot encode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Full preprocessor
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features_ohe),
], remainder='drop')  # drops neighborhood and any leftover cols

print('Preprocessor built.')

In [ ]:
# Apply feature engineering to train features before passing to pipeline
feature_adder = MoroccanHousingFeatureAdder()
housing_features_eng = feature_adder.transform(housing_features)

# Quick check — preprocessor output shape
X_sample = preprocessor.fit_transform(housing_features_eng, housing_labels)
print(f'Preprocessed training shape: {X_sample.shape}')

## 10. Select and Train Models

We evaluate 3 models exactly as Géron does:
1. Linear Regression (baseline)
2. Decision Tree Regressor
3. Random Forest Regressor

In [ ]:
def build_pipeline(model):
    return Pipeline([
        ('feature_adder', MoroccanHousingFeatureAdder()),
        ('preprocessor',  preprocessor),
        ('model',         model),
    ])

def rmse_cv(pipeline, X, y, cv=10):
    """Return cross-validated RMSE scores."""
    scores = cross_val_score(pipeline, X, y,
                             scoring='neg_mean_squared_error', cv=cv)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores

def display_scores(scores, label):
    print(f'\n=== {label} ===')
    print(f'  RMSE scores: {np.round(scores, 0)}')
    print(f'  Mean:  {scores.mean():,.0f} MAD')
    print(f'  Std:   {scores.std():,.0f} MAD')

### 10.1 Linear Regression

In [ ]:
lin_pipeline = build_pipeline(LinearRegression())
lin_scores = rmse_cv(lin_pipeline, housing_features, housing_labels)
display_scores(lin_scores, 'Linear Regression')

### 10.2 Decision Tree Regressor

In [ ]:
tree_pipeline = build_pipeline(DecisionTreeRegressor(random_state=42))
tree_scores = rmse_cv(tree_pipeline, housing_features, housing_labels)
display_scores(tree_scores, 'Decision Tree')

### 10.3 Random Forest Regressor

In [ ]:
forest_pipeline = build_pipeline(RandomForestRegressor(n_estimators=100, random_state=42))
forest_scores = rmse_cv(forest_pipeline, housing_features, housing_labels)
display_scores(forest_scores, 'Random Forest')

In [ ]:
# Visual comparison
model_names  = ['Linear Regression', 'Decision Tree', 'Random Forest']
all_scores   = [lin_scores, tree_scores, forest_scores]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(all_scores, labels=model_names)
ax.set_title('Cross-Validated RMSE by Model (10-fold)', fontsize=13)
ax.set_ylabel('RMSE (MAD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 11. Fine-Tune the Best Model — RandomizedSearchCV

Like Géron, we use `RandomizedSearchCV` on the Random Forest.

In [ ]:
from scipy.stats import randint

param_dist = {
    'model__n_estimators':      randint(50, 300),
    'model__max_features':      ['sqrt', 'log2', None],
    'model__max_depth':         [None, 10, 20, 30, 40],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf':  randint(1, 10),
}

rnd_search = RandomizedSearchCV(
    forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rnd_search.fit(housing_features, housing_labels)

best_params = rnd_search.best_params_
best_rmse   = np.sqrt(-rnd_search.best_score_)

print(f'\nBest params: {best_params}')
print(f'Best CV RMSE: {best_rmse:,.0f} MAD')

In [ ]:
# Feature importances from the best model
best_model = rnd_search.best_estimator_

# Get feature names from preprocessor
ohe_categories = best_model.named_steps['preprocessor'].named_transformers_['cat']['onehot'].categories_
cat_feature_names = []
for cat_col, cats in zip(cat_features_ohe, ohe_categories):
    cat_feature_names.extend([f'{cat_col}_{c}' for c in cats])

all_feature_names = num_features + cat_feature_names

importances = best_model.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

# Plot top 20
fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.head(20).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 12. Evaluate on the Test Set

In [ ]:
# Prepare test set
X_test = strat_test.drop(columns=['price_dh'])
y_test = strat_test['price_dh'].copy()

# Predict with best model
y_pred = best_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Test RMSE: {test_rmse:,.0f} MAD')
print(f'Test RMSE: {test_rmse/1_000_000:.2f}M MAD')

# As % of median price
median_price = y_test.median()
print(f'Median test price: {median_price:,.0f} MAD')
print(f'RMSE as % of median: {test_rmse / median_price * 100:.1f}%')

In [ ]:
# 95% confidence interval on test RMSE using bootstrapping
from scipy import stats

def bootstrap_rmse_ci(y_true, y_pred, n_bootstrap=1000, ci=0.95):
    np.random.seed(42)
    n = len(y_true)
    rmses = []
    for _ in range(n_bootstrap):
        idx = np.random.randint(0, n, n)
        rmse = np.sqrt(mean_squared_error(y_true.iloc[idx], y_pred[idx]))
        rmses.append(rmse)
    lower = np.percentile(rmses, (1 - ci) / 2 * 100)
    upper = np.percentile(rmses, (1 + ci) / 2 * 100)
    return lower, upper

lo, hi = bootstrap_rmse_ci(y_test, y_pred)
print(f'95% CI on Test RMSE: [{lo:,.0f} — {hi:,.0f}] MAD')

In [ ]:
# Actual vs Predicted scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test / 1_000_000, y_pred / 1_000_000, alpha=0.4, edgecolors='none', s=25)
max_val = max(y_test.max(), y_pred.max()) / 1_000_000
ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual Price (millions MAD)')
ax.set_ylabel('Predicted Price (millions MAD)')
ax.set_title('Actual vs. Predicted — Test Set (Random Forest)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(y_pred / 1_000_000, residuals / 1_000_000, alpha=0.4, edgecolors='none', s=25)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicted Price (millions MAD)')
ax.set_ylabel('Residual (millions MAD)')
ax.set_title('Residuals vs. Predicted — Test Set')
plt.tight_layout()
plt.show()

## 13. Summary

| Model | CV RMSE (mean) | CV RMSE (std) |
|---|---|---|
| Linear Regression | ... | ... |
| Decision Tree | ... | ... |
| Random Forest (base) | ... | ... |
| **Random Forest (tuned)** | **...** | **...** |

_(Fill in after running the notebook)_

### Next Steps
- Add **neighborhood target encoding** (too many categories for OHE)
- Try **XGBoost / LightGBM** for better performance on tabular data
- Collect **latitude/longitude** per neighborhood for geographic features
- Handle **price outliers** more carefully (e.g., 68M MAD villa)
- Add **log-transform** on `price_dh` — the distribution is right-skewed